# P1: Data Generation — Phase 1
## ATRD: Adaptive Test-Time Reasoning Distillation

**Baseline evaluation → Failure extraction → Synthetic generation → Filtering → Deduplication → Mixing**

- Model: `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16`
- Deliverable: `final_train_dataset.jsonl`

In [ ]:
# Cell 1: Imports + Reproducibility Setup
import random
import numpy as np
import torch
import os, sys, json, re, hashlib
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Configuration
from dataclasses import dataclass

@dataclass(frozen=True)
class Phase1Config:
    BASE_MODEL: str = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16"
    MAX_TOKENS: int = 7680
    TEMPERATURE: float = 0.0
    BENCHMARK_PATH: str = "/kaggle/input/nemotron-benchmark"
    OUTPUT_DIR: Path = Path("/kaggle/working")
    RAW_SYNTHETIC_PATH: str = "/kaggle/working/raw_synthetic_dataset.jsonl"
    FILTERED_PATH: str = "/kaggle/working/filtered_synthetic_dataset.jsonl"
    FINAL_PATH: str = "/kaggle/working/final_train_dataset.jsonl"
    SYNTHETIC_TARGET: int = 10000
    NUM_FAILURE_MODES: int = 5
    API_MODEL: str = "deepseek-r1"
    API_TEMPERATURE: float = 0.7

CFG = Phase1Config()
print(f"Phase 1 Config: model={CFG.BASE_MODEL}")
print(f"Synthetic target: {CFG.SYNTHETIC_TARGET:,} problems")

In [ ]:
# Cell 3: Helper Functions

def format_prompt(question: str) -> str:
    """Format question with thinking instruction tokens."""
    return (
        f"{question}\n"
        "<<thinking>>\n"
        "[reasoning trace]\n"
        "</thinking>>\n"
        "Answer: \\boxed{}"
    )


def extract_boxed_answer(text: str) -> str:
    """Extract answer from \\boxed{} format with nested brace support."""
    pattern = r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}'
    matches = re.findall(pattern, text)
    return matches[-1].strip() if matches else ''


def check_answer(predicted: str, expected: str, tolerance: float = 0.01) -> bool:
    """Check if predicted matches expected within tolerance."""
    try:
        return abs(float(predicted) - float(expected)) <= tolerance
    except (ValueError, TypeError):
        return predicted.strip() == expected.strip()


def classify_failure(response: Dict[str, Any]) -> str:
    """Classify failure type in model response."""
    answer = response.get('answer', '')
    reasoning = response.get('reasoning', '')
    if not answer:
        return 'no_answer'
    if not reasoning:
        return 'incomplete'
    if '\\boxed' not in answer:
        return 'format_error'
    return 'wrong_answer'


print('Helper functions loaded: format_prompt, extract_boxed_answer, check_answer, classify_failure')

In [ ]:
# Cell 4: Load Base Model
from src.models.loader import ModelLoader

loader = ModelLoader()
model = loader.load_model(quantize=True)
tokenizer = loader.load_tokenizer()

if hasattr(model, 'num_parameters'):
    print(f"Model loaded: {model.num_parameters():,} params")
else:
    print(f"Model loaded: {type(model).__name__}")

In [ ]:
# Cell 5: Baseline Evaluation
import time
from pathlib import Path

# Load benchmark
benchmark_path = Path(CFG.BENCHMARK_PATH)
problems = []
if benchmark_path.exists():
    for f_path in sorted(benchmark_path.glob("*.jsonl")):
        with open(f_path) as f:
            for line in f:
                problems.append(json.loads(line))
    print(f"Loaded {len(problems)} benchmark problems")
else:
    problems = [
        {"question": "Solve x + 5 = 10", "answer": "5"},
        {"question": "What is 2+2?", "answer": "4"},
    ]
    print(f"Benchmark not found, using {len(problems)} inline test problems")

# Run inference
baseline_results = []
for i, prob in enumerate(problems):
    prompt = format_prompt(prob["question"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=CFG.MAX_TOKENS,
        temperature=CFG.TEMPERATURE,
        top_p=1.0,
    )
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    extracted = extract_boxed_answer(raw_output)
    correct = check_answer(extracted, prob["answer"])
    failure = classify_failure({"answer": raw_output, "reasoning": raw_output})
    baseline_results.append({
        "question": prob["question"],
        "expected": prob["answer"],
        "predicted": extracted,
        "correct": correct,
        "failure_mode": failure,
    })
    if (i + 1) % 50 == 0:
        print(f"  Evaluated {i+1}/{len(problems)}")

correct_count = sum(1 for r in baseline_results if r["correct"])
accuracy = correct_count / max(len(baseline_results), 1)
print(f"\nBaseline accuracy: {accuracy:.2%} ({correct_count}/{len(baseline_results)})")

# Save results
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(CFG.OUTPUT_DIR / "baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
print(f"Saved baseline_results.json")

In [ ]:
# Cell 6: Failure Mode Analysis
from collections import Counter

failure_counts = Counter(r["failure_mode"] for r in baseline_results)
print("Failure Mode Distribution:")
print(f"  {'Mode':<25} {'Count':>8} {'Rate':>8}")
print("  " + "-" * 43)
for mode, count in failure_counts.most_common():
    rate = count / max(len(baseline_results), 1)
    bar = "#" * int(rate * 40)
    print(f"  {mode:<25} {count:>8} {rate:>7.1%} {bar}")

# Collect failure examples per mode
failure_examples: Dict[str, List[Dict]] = {}
for r in baseline_results:
    if not r["correct"]:
        mode = r["failure_mode"]
        failure_examples.setdefault(mode, []).append(r)

for mode, examples in failure_examples.items():
    print(f"  {mode}: {len(examples)} examples")

# Save failure modes
with open(CFG.OUTPUT_DIR / "failure_modes.json", "w") as f:
    serializable = {k: v for k, v in failure_examples.items()}
    json.dump(serializable, f, indent=2)
print(f"Saved failure_modes.json with {len(failure_examples)} categories")

In [ ]:
# Cell 7: Synthetic Data Generation
import time
from src.data.synthetic_generator import SyntheticGenerator, FAILURE_MODE_DESCRIPTIONS

gen = SyntheticGenerator(
    config_path="configs/competition_params.json",
    output_dir=str(CFG.OUTPUT_DIR / "synthetic"),
    api_key=os.environ.get("API_KEY", ""),
    primary_model=CFG.API_MODEL,
)

raw_synthetic = []
problems_per_mode = CFG.SYNTHETIC_TARGET // CFG.NUM_FAILURE_MODES

for mode in FAILURE_MODE_DESCRIPTIONS:
    if mode not in failure_examples:
        print(f"  Skipping {mode}: no failure examples")
        continue
    print(f"\nGenerating for failure mode: {mode}")
    batch = gen._generate_for_mode(
        mode=mode,
        description=FAILURE_MODE_DESCRIPTIONS[mode],
        examples=failure_examples[mode],
        target_count=problems_per_mode,
    )
    raw_synthetic.extend(batch)
    print(f"  Total so far: {len(raw_synthetic)} / {CFG.SYNTHETIC_TARGET}")

gen.save_dataset(raw_synthetic, filename="raw_synthetic_dataset.jsonl")
print(f"\nSynthetic generation complete: {len(raw_synthetic)} problems")

In [ ]:
# Cell 8: Quality Filtering (LLM-as-Judge)
from src.data.judge_filter import JudgeFilter

judge = JudgeFilter(threshold=0.80)
filtered = judge.filter_dataset(raw_synthetic)
report = judge.generate_report(raw_synthetic, filtered)

# Save filtered
with open(CFG.OUTPUT_DIR / "filtered_synthetic_dataset.jsonl", "w") as f:
    for ex in filtered:
        f.write(json.dumps(ex) + "\n")
print(f"Saved {len(filtered)} filtered examples to filtered_synthetic_dataset.jsonl")
print(f"Filter report: pass_rate={report['pass_rate']:.1%}")

In [ ]:
# Cell 9: Deduplication (MinHash + LSH)
from src.data.deduplicator import Deduplicator

dedup = Deduplicator(similarity_threshold=0.85)
deduplicated = dedup.deduplicate(filtered, key="question")
print(f"After dedup: {len(deduplicated)} examples")

In [ ]:
# Cell 10: Dataset Mixing (50/25/25)
from src.data.dataset_mixer import DatasetMixer

mixer = DatasetMixer(seed=SEED)

# Simulate/reference OpenMathReasoning and OpenCodeReasoning datasets
openmath_data = [{"question": f"OpenMath Q{i}", "answer": str(i)} for i in range(1000)]
opencode_data = [{"question": f"OpenCode Q{i}", "answer": str(i)} for i in range(1000)]

final_dataset = mixer.mix(
    synthetic=deduplicated,
    math_reasoning=openmath_data,
    code_reasoning=opencode_data,
    max_total=50000,
    failure_mode_ratios=None,
)

dist = mixer.get_distribution(final_dataset)
print(f"Final dataset size: {len(final_dataset)}")
print(f"Source distribution: {dist}")

In [ ]:
# Cell 11: Leakage Check
from src.data.dataset_mixer import check_leakage

train_texts = [json.dumps(ex) for ex in final_dataset]
test_texts = [json.dumps(prob) for prob in problems]

overlap = check_leakage(train_texts, test_texts, n=5)
if overlap == 0:
    print("\n✓ No leakage detected. Test set is clean.")
else:
    print(f"\n⚠ Leakage detected: {overlap} n-gram matches.")

In [ ]:
# Cell 12: Save & Upload to Kaggle Datasets
mixer.save_mixed(final_dataset, output_path=str(CFG.FINAL_PATH))
print(f"Final training dataset saved to {CFG.FINAL_PATH}")

# Optional: version to Kaggle Datasets
import subprocess
dataset_slug = "samar/atrd-final-train-dataset"
try:
    result = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", str(CFG.OUTPUT_DIR),
         "-m", "Phase 1: Final training dataset with 50/25/25 ratio"],
        capture_output=True, text=True, timeout=30,
    )
    print(f"Kaggle Dataset versioned: {result.stdout}")
except Exception as e:
    print(f"Kaggle upload skipped (not configured): {e}")

# Statistics report
stats = {
    "baseline_accuracy": accuracy,
    "failure_distribution": dict(failure_counts),
    "raw_synthetic_count": len(raw_synthetic),
    "filtered_count": len(filtered),
    "dedup_removed": len(filtered) - len(deduplicated),
    "final_count": len(final_dataset),
    "source_distribution": dist,
}
with open(CFG.OUTPUT_DIR / "logs" / "p1_baseline_eval.json", "w") as f:
    json.dump(stats, f, indent=2)
print(f"Saved statistics to logs/p1_baseline_eval.json")

In [ ]:
# Cell 13: Cleanup
import gc
del model, tokenizer, loader
torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared.")
print("\nPhase 1 complete. Run: python scripts/verify_unit_completion.py P1")